# E17 — Robustness, consistency & gap-closure consolidation

**Spec:** `EXPERIMENT_PLAN.md` §E17 (revised 2026-09-21). **Decisions:** `DECISIONS.md`, the
2026-09-21 entries on H1's learner scope and on the coverage-restoration criterion. Three parts:

- **H1 — robustness** *(scoped to **persistence, GBM and GRU**; MC-dropout / E8 is excluded — see
  §2a)*. Are the headline results stable under reasonable alternative statistical choices?
  (i) the bootstrap's clustering assumption (Q-STAT-03c), via a **mission-level cluster
  bootstrap**; (ii) the weight-clipping cap (Q-SEL-03) — a *confirmation that clipping stays
  untriggered under stricter triggers*, not a live sweep; (iii) the declared multiple-comparison
  policy (Q-STAT-04), applied to the family of p-values E11 already recorded.
- **H2 — gap closure.** The **one-sided CQR** coverage cell, completing the
  {split, CQR} × {two-sided, one-sided} restoration matrix. The only genuinely new scientific
  quantity in E17.
- **H3 — synthesis.** Is there a shared *per-event* mechanism behind the five manifestations of
  the train/test high-risk imbalance? An honest null is a valid outcome.

> **The coverage-restoration criterion (corrected 2026-09-21).** A conformal guarantee is
> **coverage ≥ 1 − α — one-sided.** A cell is **RESTORED** when the naive arm's CI upper bound lies
> below nominal (under-coverage established) and the rule-weighted arm's CI upper bound lies at or
> above it (under-coverage no longer established). The original implementation instead required
> nominal to lie *inside* the weighted CI, which marks a conservative, over-covering arm as a
> failure — a specification bug. **Both are reported: the corrected criterion is primary; the
> original is a labelled secondary comparison.**

> **No new formal test is introduced** (Q-STAT-04). H2 inherits E11/E12's coverage convention
> exactly; H3 is descriptive association only.

In [ ]:
# Papermill parameters. RECOMPUTE=False re-renders from the tables a completed E17 run already
# wrote, re-deriving only the verdicts (used for the 2026-09-21 criterion correction, which
# changes which existing quantity defines "restored" and recomputes nothing).
RECOMPUTE = True

In [ ]:
# --- Setup + provenance (invariant I4) ------------------------------------------------------
import hashlib, json, subprocess, sys
from datetime import datetime, timezone

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from kelvins_conformal.config import REPO_ROOT, load_config
from kelvins_conformal.models import robustness_runner as RR
from kelvins_conformal.reporting import write_table_atomic
from kelvins_conformal.robustness import NO_DEFICIT, NOT_RESTORED, RESTORED

cfg = load_config()
PREFIX = "e17_"
FIGDIR = cfg.path("figures_dir"); FIGDIR.mkdir(parents=True, exist_ok=True)
TABDIR = cfg.path("tables_dir"); TABDIR.mkdir(parents=True, exist_ok=True)

def git_sha():
    try:
        return subprocess.run(["git", "rev-parse", "HEAD"], cwd=str(REPO_ROOT),
                              capture_output=True, text=True, check=True).stdout.strip()
    except Exception:
        return "UNAVAILABLE"

def save_table(df, name):
    write_table_atomic(df, TABDIR / f"{PREFIX}{name}.csv"); print(f"saved: reports/tables/{PREFIX}{name}.csv")

def save_fig(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(FIGDIR / f"{PREFIX}{name}.{ext}", dpi=160, bbox_inches="tight", facecolor=fig.get_facecolor())
    print(f"saved: reports/figures/{PREFIX}{name}.png|pdf")

SURFACE, INK, INK2, MUTED, GRID, AXIS = "#fcfcfb", "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7"
GREEN, RED = "#1baf7a", "#d6453a"
plt.rcParams.update({"axes.edgecolor": AXIS, "axes.labelcolor": INK2, "xtick.color": MUTED,
                     "ytick.color": MUTED, "text.color": INK, "axes.titlecolor": INK})

## 1. Run — or re-render from the computed tables

In [ ]:
if RECOMPUTE:
    RES = RR.run_e17(cfg)
    for key in RR.COMPUTED_TABLES:
        save_table(RES[key], key)
else:
    # Recompute nothing: load the computed tables and re-derive the verdicts through the same
    # pure functions run_e17 uses. The computed tables are NOT rewritten, so their bytes stay
    # exactly as the computing run left them.
    RES = RR.load_e17(cfg)
for key in RR.DERIVED_TABLES:
    save_table(RES[key], key)
meta = RES["meta"]
PRIM = meta["primary_level"]
PROVENANCE = {"experiment_ids": ["E17"], "analysis": "robustness, consistency & gap-closure consolidation",
              "recomputed": meta["recomputed"], "rendered_git_sha": git_sha(), "config_hash": cfg.config_hash,
              "rendered_utc": datetime.now(timezone.utc).isoformat(), "python": sys.version.split()[0],
              "computation_timings_s": meta["timings"],
              "restoration_criterion": "primary: coverage >= nominal (one-sided guarantee, CI upper bound); "
                                       "secondary: CI contains nominal (original, a specification bug)"}
if not meta["recomputed"]:
    PROVENANCE["source_table_sha256"] = meta["source_table_sha256"]
print(json.dumps(PROVENANCE, indent=2, default=str))

## 2a. Scope of H1 — which learners are checked, and which is not

**MC-dropout (the E8 Bayesian arm) is NOT included in H1's robustness checks.** This is a disclosed
scope boundary, decided by Sidh on 2026-09-21 before E17 was re-run — not an oversight and not a
silent gap.

**Why.** No `e8_mcdropout` hyperparameter cache exists under the current config hash. E8 published
under an *older* hash — the configuration has since gained the `decision_cost` and
`threshold_analysis` blocks — so a fresh search now would select hyperparameters **E8 never used**.
E17's Bayesian arm would then be a different fitted model from the one E8 reported, and every
cross-reference between this supplement and E8's findings would compare two models while appearing
to compare one.

**What this does and does not limit.** The Gate-2 headline contrast — the project's only formally
tested result — is **persistence**, which has no hyperparameters and needs no search; it is inside
the retained scope. **H2 and H3 are unaffected.** **E8's own published results are unchanged.**

In [ ]:
display(RES["h1_excluded_learners"])
print("H1 covers:", meta["h1"]["h1_learners"])
print("H1 excludes:", meta["h1"]["h1_excluded_learners"])

## 2. H1 (i) — clustering assumption: mission-level cluster bootstrap vs the event bootstrap (Q-STAT-03c)

The **same** per-event covered indicators feed both schemes; only the resampling differs. A width
ratio above 1 means the cluster bootstrap is wider — the expected direction when events within a
mission are positively correlated.

In [ ]:
cb = RES["h1_cluster_bootstrap"]
print(f"clusters: {meta['h1']['n_clusters']} missions over {meta['h1']['n_supported_events']} supported events; "
      f"largest cluster holds {100*cb['largest_cluster_frac'].iloc[0]:.1f}% of events")
head = cb[(cb["nominal"] == PRIM) & (cb["sided"] == "two")]
display(pd.DataFrame({
    "learner": head["learner"], "method": head["method"], "coverage": head["coverage"].round(4),
    "iid 95% CI": [f"[{r.iid_lo:.4f}, {r.iid_hi:.4f}]" for r in head.itertuples()],
    "cluster 95% CI": [f"[{r.cluster_lo:.4f}, {r.cluster_hi:.4f}]" for r in head.itertuples()],
    "width ratio (cluster/iid)": head["width_ratio"].round(2),
}).reset_index(drop=True))
N_WIDER = int((cb["width_ratio"] > 1).sum())
print(f"\nwidth ratio across all {len(cb)} arms: median {cb['width_ratio'].median():.2f}, "
      f"range [{cb['width_ratio'].min():.2f}, {cb['width_ratio'].max():.2f}]; cluster wider in {N_WIDER} of {len(cb)}")

### Does the Gate-2 conclusion survive the clustering assumption? Both criteria, corrected primary

Each row carries both CI bounds, so every verdict is auditable from its own row.

In [ ]:
v = RES["h1_gate2_verdict"]
display(pd.DataFrame({
    "learner": v["learner"], "nominal": v["nominal"], "scheme": v["scheme"],
    "naive cov": v["naive_coverage"].round(4),
    "naive CI": [f"[{r.naive_lo:.4f}, {r.naive_hi:.4f}]" for r in v.itertuples()],
    "weighted cov": v["weighted_coverage"].round(4),
    "weighted CI": [f"[{r.weighted_lo:.4f}, {r.weighted_hi:.4f}]" for r in v.itertuples()],
    "PRIMARY: coverage >= nominal": v["verdict"],
    "secondary: CI contains nominal": v["verdict_containment"],
    "weighted over-covers": v["weighted_over_covers"],
}).reset_index(drop=True))
AGREE = {c: RR.scheme_agreement(v, c) for c in ("verdict", "verdict_containment")}
for c, label in (("verdict", "PRIMARY (coverage >= nominal)"), ("verdict_containment", "secondary (CI contains nominal)")):
    a = AGREE[c]
    print(f"{label:34}: iid and cluster agree in {int(a['agree'].sum())} of {len(a)} learner x level cells; "
          f"disagree at {[(r.learner, r.nominal) for r in a[~a['agree']].itertuples()] or 'none'}")

### The Gate-2 headline, under each criterion

Persistence, two-sided, at the primary level — the project's only formally tested contrast.

In [ ]:
H = v[(v["learner"] == "persistence") & (v["nominal"] == PRIM)]
for r in H.itertuples():
    print(f"{r.scheme:8} naive {r.naive_coverage:.4f} CI [{r.naive_lo:.4f}, {r.naive_hi:.4f}] | "
          f"weighted {r.weighted_coverage:.4f} CI [{r.weighted_lo:.4f}, {r.weighted_hi:.4f}] | "
          f"PRIMARY {r.verdict} | secondary {r.verdict_containment}")
HEADLINE_OK = bool((H["verdict"] == RESTORED).all())
HEADLINE_STRONG = bool(H["weighted_ci_wholly_above_nominal"].all())
print(f"\nPRIMARY verdict: {'RESTORED under both schemes - the headline PASSES the clustering check' if HEADLINE_OK else 'NOT restored under at least one scheme'}.")
print(f"Weighted CI wholly above nominal under both schemes (strong restoration, not a width artifact): {HEADLINE_STRONG}")
print(f"Under the secondary (containment) criterion the same headline reads: {sorted(set(H['verdict_containment']))} "
      f"- because the weighted arm over-covers ({bool(H['weighted_over_covers'].all())}).")

### Genuine fragility — reported as-is, not rounded into either column

Where the primary verdict differs between the two resampling schemes, the cell is **not** folded
into the majority. The mechanism matters for reading the agreement figure above: the primary
criterion asks whether under-coverage **can still be established**, so a *less precise* interval
makes RESTORED *easier* to reach. The cluster bootstrap is the wider of the two, so it leans toward
RESTORED. A disagreement of that kind is evidence of fragility, not of restoration.

In [ ]:
FRAGILE = AGREE["verdict"][~AGREE["verdict"]["agree"]]
if FRAGILE.empty:
    print("No primary-criterion disagreement between the iid and cluster bootstraps.")
for f in FRAGILE.itertuples():
    rows = v[(v["learner"] == f.learner) & (v["nominal"] == f.nominal)].set_index("scheme")
    pt = float(rows["weighted_coverage"].iloc[0])
    print(f"FRAGILE: {f.learner}, two-sided, nominal {f.nominal:.0%}")
    for s in ("iid", "cluster"):
        r = rows.loc[s]
        print(f"  {s:8} naive CI upper {r.naive_hi:.4f} | weighted CI [{r.weighted_lo:.4f}, {r.weighted_hi:.4f}] "
              f"-> {r.verdict}")
    wider = "cluster" if (rows.loc["cluster", "weighted_hi"] - rows.loc["cluster", "weighted_lo"]) > \
                          (rows.loc["iid", "weighted_hi"] - rows.loc["iid", "weighted_lo"]) else "iid"
    print(f"  weighted point estimate {pt:.4f} is {'BELOW' if pt < f.nominal else 'at or above'} nominal "
          f"{f.nominal:.2f} under both schemes (they share one point estimate).")
    print(f"  The verdict changes only because the {wider} interval is wider and can no longer reject "
          f"under-coverage - not because coverage improved. Reported as a fragility.")

### Which verdicts did the criterion correction change?

A check on the correction itself: a fix for *this specific bug* must change exactly the
over-covering arms and nothing else.

In [ ]:
CH = v[~v["criteria_agree"]]
print(f"verdicts changed by the correction: {len(CH)} of {len(v)}")
print(f"  every changed verdict is an over-covering arm: {bool(CH['weighted_over_covers'].all())}")
print(f"  every change is toward RESTORED: {bool((CH['verdict'] == RESTORED).all())}")
print(f"  any under-covering arm's verdict changed: {bool((~CH['weighted_over_covers']).any())}")
display(CH[["learner", "nominal", "scheme", "weighted_coverage", "verdict_containment", "verdict"]].round(4).reset_index(drop=True))

## 3. H1 (ii) — weight-clipping cap (Q-SEL-03)

Q-SEL-03 Decision B makes clipping **conditional**: it fires only when Pareto k̂ exceeds the trigger.
This confirms it still does not fire at progressively stricter triggers, and reports the induced
bias Δ_B clipping *would* introduce (0 when untriggered).

In [ ]:
clip = RES["h1_clipping"]
display(clip[["weights", "khat", "khat_band", "khat_trigger", "clipping_triggered",
              "clipped_fraction", "bias_delta", "n_calibration", "n_effective"]].round(4).reset_index(drop=True))
print("\nClipping triggered anywhere:", bool(clip["clipping_triggered"].any()))
for w in clip["weights"].unique():
    sub = clip[clip["weights"] == w]
    k, ne, n = sub["khat"].iloc[0], sub["n_effective"].iloc[0], sub["n_calibration"].iloc[0]
    print(f"  {w}: k-hat = {k:.3f} ({sub['khat_band'].iloc[0]}); effective n = {ne:.1f} of {n} ({100*ne/n:.1f}%)")

## 4. H1 (iii) — multiple-comparison policy (Q-STAT-04)

Holm–Bonferroni applied to the family of p-values E11 **already recorded**. A sweep of every
committed table finds that family has exactly one member, so Holm is the identity: the check
confirms the declared policy was **adhered to**, which is only checkable because no other formal
test was ever run.

In [ ]:
mc = RES["h1_multiple_comparison"]
pcol = meta["h1"]["p_value_column"]
cols = [c for c in ("learner", "method", "sided", "nominal") if c in mc.columns]
display(mc[cols + [pcol, "p_adjusted_holm", "rejected_holm", "n_tests_in_family"]].reset_index(drop=True))
print(f"\nfamily size {mc['n_tests_in_family'].iloc[0]}; rejected at alpha=0.05 after Holm: "
      f"{int(mc['rejected_holm'].sum())} of {len(mc)}")
print(f"raw p = {mc[pcol].min():.3g}; Holm-adjusted p = {mc.loc[mc[pcol].idxmin(), 'p_adjusted_holm']:.3g}; "
      f"still rejected = {bool(mc.loc[mc[pcol].idxmin(), 'rejected_holm'])}")

## 5. H2 — the completed coverage-restoration matrix (the new cell)

Three cells are reloaded from the committed E9/E11 and E12 tables — exactly the numbers those
experiments published. The fourth, **CQR one-sided**, is computed by E17 for the first time. Both
criteria are shown; in this matrix they agree in every cell.

In [ ]:
mx = RES["coverage_restoration_matrix"]
display(mx[["family", "sided", "source", "naive_coverage", "naive_ci", "weighted_coverage",
            "weighted_ci", "change_pp", "verdict", "verdict_containment"]].round(4).reset_index(drop=True))
print(f"\nThe two criteria agree in all four matrix cells: {bool(mx['criteria_agree'].all())}")
print("\nH2 cell (CQR, one-sided), all levels:")
h2 = RES["h2_coverage"]
display(h2[["method", "nominal", "coverage_mean", "coverage_sd", "n", "cp_lo_mean", "cp_hi_mean",
            "gap_pp", "frac_inf_width"]].round(4).reset_index(drop=True))

In [ ]:
# The 2x2 matrix as a figure (primary criterion): where the selection-bias correction works.
fig, ax = plt.subplots(figsize=(7.2, 4.2), facecolor=SURFACE)
ax.set_facecolor(SURFACE)
fams, sides = ["split conformal", "CQR"], ["two", "upper"]
label = {"two": "two-sided", "upper": "one-sided (upper)"}
for i, fam in enumerate(fams):
    for j, sd in enumerate(sides):
        r = mx[(mx["family"] == fam) & (mx["sided"] == sd)].iloc[0]
        restored = r["verdict"] == RESTORED
        colour = GREEN if restored else (MUTED if r["verdict"] == NO_DEFICIT else RED)
        ax.add_patch(plt.Rectangle((j, -i), 1, 1, facecolor=colour, alpha=0.16, edgecolor=AXIS, lw=1.2))
        ax.text(j + 0.5, -i + 0.72, r["verdict"], ha="center", va="center", fontsize=11,
                color="#0a7a52" if restored else colour, fontweight="bold")
        ax.text(j + 0.5, -i + 0.46, f"naive {r['naive_coverage']:.3f} -> weighted {r['weighted_coverage']:.3f}",
                ha="center", va="center", fontsize=9, color=INK)
        ax.text(j + 0.5, -i + 0.26, f"{r['change_pp']:+.1f} pp   ({r['source']})",
                ha="center", va="center", fontsize=8, color=INK2)
ax.set_xlim(0, 2); ax.set_ylim(-1, 1)
ax.set_xticks([0.5, 1.5]); ax.set_xticklabels([label[s] for s in sides], fontsize=10)
ax.set_yticks([0.5, -0.5]); ax.set_yticklabels(fams, fontsize=10)
ax.tick_params(length=0)
for s in ax.spines.values():
    s.set_visible(False)
ax.set_title(f"Does rule-derived weighting restore coverage? Official test set, nominal {PRIM:.0%}",
             loc="left", fontsize=11)
fig.text(0.0, -0.04, "Each cell: naive -> rule-weighted empirical coverage (GBM), and the change in percentage "
         "points. RESTORED: the naive CI's upper bound lies below nominal and the weighted CI's upper bound "
         "lies at or above it (the one-sided guarantee, coverage >= 1 - alpha).",
         fontsize=7.5, color=INK2, ha="left", va="top", wrap=True)
fig.tight_layout()
save_fig(fig, "coverage_restoration_matrix")
plt.close(fig)

## 6. H3 — is there one shared per-event mechanism behind the five manifestations?

**Stated before looking:** only three of the five manifestations are event-level quantities at all.
M4 (grid-instrument distortion) and M5 (the asymmetric partial fix) are properties of the threshold
*grid* and of a selection criterion over it — they have no per-event membership. A per-event
diagnostic therefore **cannot**, even in principle, unify all five; the most it can do is unify
M1–M3. H3 does not use the restoration criterion.

> **Construction caveat — read before the numbers below.** The candidate diagnostic is the
> per-event residual d = y − ŷ of the GBM point prediction. **All three event-level memberships
> are defined as functions of that same residual**:
> - **M1** is `d ≥ 90th percentile of d` — a threshold on d itself;
> - **M3** is `high-risk and ŷ < −6`; high-risk means y ≥ −6 > ŷ, so every M3 member has d > 0
>   *by definition*;
> - **M2** is "uncovered by the two-sided interval ŷ ± Q", which holds exactly when |d| > Q — a
>   function of |d| (per seed).
>
> The associations and overlap lifts below are therefore largely **consequences of these
> definitions**, not evidence that one per-event mechanism drives the manifestations. As
> constructed, this test cannot distinguish a *shared mechanism* from a *shared definition*, and
> it is **not** read as support for a unification. Per E17's specification this falls back to the
> honest outcome: the manifestations are related in effect, and this analysis does not show them
> to reduce to one per-event mechanism. A non-circular test — a diagnostic not used to define
> membership — is a design question left for Sidh.

In [ ]:
assoc = RES["h3_association"]
display(assoc[["manifestation", "description", "level_declared", "n_members",
               "diagnostic_mean_in", "diagnostic_mean_out", "high_risk_share_in",
               "high_risk_share_out", "point_biserial_r"]].round(4))
ov = RES["h3_overlap"]
display(ov.round(3))
ev = assoc[assoc["level_declared"] == "event"]
R_MIN, R_MAX = float(ev["point_biserial_r"].abs().min()), float(ev["point_biserial_r"].abs().max())
LIFT_MIN, LIFT_MAX = float(ov["lift"].min()), float(ov["lift"].max())
print(f"\nDiagnostic: {meta['h3']['diagnostic']} over {meta['h3']['n_events']} supported events.")
print(f"|point-biserial r| across M1-M3: {R_MIN:.3f} to {R_MAX:.3f}; pairwise overlap lift: {LIFT_MIN:.2f}x to {LIFT_MAX:.2f}x")
# Checks the construction caveat against the table, rather than only asserting it.
m3 = assoc[assoc["manifestation"] == "M3"].iloc[0]
m1 = assoc[assoc["manifestation"] == "M1"].iloc[0]
print()
print(f"Construction checks: M3 high-risk share = {m3['high_risk_share_in']:.3f} (1.000 by definition); "
      f"M3 mean d = {m3['diagnostic_mean_in']:.3f} (> 0 by definition); "
      f"M1 mean d in/out = {m1['diagnostic_mean_in']:.3f} / {m1['diagnostic_mean_out']:.3f} (M1 is a threshold on d).")
print("=> These associations follow from how membership is defined; they are NOT evidence of a shared mechanism.")

## 7. Summary — measurement only

In [ ]:
restored = mx[mx["verdict"] == RESTORED]
not_restored = mx[mx["verdict"] == NOT_RESTORED]
cqr_up = mx[(mx.family == "CQR") & (mx.sided == "upper")].iloc[0]
ag_p, ag_s = AGREE["verdict"], AGREE["verdict_containment"]
print(f"""
E17 - ROBUSTNESS, CONSISTENCY & GAP-CLOSURE (measurement only, exactly as observed)
{'Re-rendered from the tables of a completed run; nothing recomputed.' if not meta['recomputed'] else 'Computed in this execution.'}

 H1 SCOPE: {meta['h1']['h1_learners']} checked; {meta['h1']['h1_excluded_learners']} EXCLUDED (no E8 hyperparameter
   cache under the current config hash - disclosed in §2a).
 H1 (i) clustering: {meta['h1']['n_clusters']} missions over {meta['h1']['n_supported_events']} supported events; cluster CI wider in
   {N_WIDER} of {len(cb)} arms (median width ratio {cb['width_ratio'].median():.2f}).
   Gate-2 headline (persistence, two-sided, {PRIM:.0%}), PRIMARY criterion: {'RESTORED under both schemes - PASSES' if HEADLINE_OK else 'NOT restored under at least one scheme'};
     weighted CI wholly above nominal under both: {HEADLINE_STRONG}.
   Scheme agreement - PRIMARY: {int(ag_p['agree'].sum())}/{len(ag_p)}; secondary (containment): {int(ag_s['agree'].sum())}/{len(ag_s)}.
   FRAGILE (primary verdict depends on the scheme): {[(r.learner, r.nominal) for r in FRAGILE.itertuples()] or 'none'}.
   Criterion correction changed {len(CH)} verdicts, all over-covering arms: {bool(CH['weighted_over_covers'].all())}.
 H1 (ii) clipping: triggered anywhere across triggers 0.7 / 0.5 / 0.3: {bool(clip['clipping_triggered'].any())}.
 H1 (iii) Holm over the recorded family of {mc['n_tests_in_family'].iloc[0]}: still rejected =
   {bool(mc.loc[mc[pcol].idxmin(), 'rejected_holm'])} (adjusted p = {mc.loc[mc[pcol].idxmin(), 'p_adjusted_holm']:.3g}).
 H2 matrix, nominal {PRIM:.0%}: RESTORED in {len(restored)} of 4 cells
   ({', '.join(f"{r.family}/{r.sided}" for r in restored.itertuples()) or 'none'}); not restored in {len(not_restored)}.
   NEW cell (CQR, one-sided): naive {cqr_up.naive_coverage:.4f} -> weighted {cqr_up.weighted_coverage:.4f}
   ({cqr_up.change_pp:+.1f} pp), {cqr_up.verdict}. Both criteria agree in all four cells: {bool(mx['criteria_agree'].all())}.
 H3: {meta['h3']['event_level_manifestations']} of 5 manifestations are event-level; M4/M5 are instrument-level.
   |r| across M1-M3 {R_MIN:.3f} to {R_MAX:.3f}; pairwise lift {LIFT_MIN:.2f}x to {LIFT_MAX:.2f}x - CIRCULAR:
   all three memberships are defined from the same residual d, so these do not evidence a shared
   mechanism. Honest-null fallback applies: related in effect, not shown to reduce to one mechanism.

 NOT DECIDED HERE: whether the completed matrix changes any manuscript claim; how the H3 outcome
 is framed; anything in E18.
""")
(cfg.path("reports_dir") / "06_robustness_provenance.json").write_text(
    json.dumps(PROVENANCE, indent=2, default=str), encoding="utf-8")
print("provenance:", cfg.path("reports_dir") / "06_robustness_provenance.json")